# MBERE — End-to-End ML Pipeline (Addis Ababa Road Traffic Accident Severity)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Terrymanzi/MBERE_ML/blob/main/ml/notebooks/ml_pipeline_end_to_end.ipynb)

This notebook runs the **actual `ml/` package** from the MBERE_ML repository end to end on Google Colab — it clones the repo and imports the real modules (`ml.preprocessing`, `ml.features`, `ml.training`, `ml.evaluation`, `ml.explainability`) rather than re-implementing the pipeline inline, so results here match the local pipeline exactly.

**Dataset:** Addis Ababa Road Traffic Accident data. **Task:** predict accident severity (`Slight` / `Serious` / `Fatal`) as an interpretable, pre-accident driver-risk proxy.

**Pipeline stages covered:**

1. Environment setup (clone + install)
2. Dataset configuration (`ml/configs/addis.yaml`)
3. Raw data exploration
4. Data cleaning (leakage-safe, deterministic)
5. Feature engineering (interpretable derived features)
6. Stratified train/test split
7. Feature selection (mutual information, train-only)
8. Encoding (ordinal + one-hot, fit on train only)
9. Authoritative preprocessing run (persists artifacts)
10. Model architecture (rule-based baseline, Random Forest, XGBoost)
11. SMOTE-in-pipeline demonstration (leak-free class rebalancing)
12. Model training (shared stratified CV harness, out-of-fold metrics)
13. Cross-validation metrics comparison
14. Held-out test set evaluation (confusion matrices, ROC curves)
15. Test metrics comparison & model selection
16. Explainability (SHAP: global summary + local waterfall)
17. Saved artifacts & reproducibility
18. Optional synthetic (Rwandan) data sanity check
19. Downloading artifacts from Colab

> **Runtime tip:** Use a CPU runtime — nothing here requires a GPU. If Colab prompts *"Restart session"* after the install cell, click **Restart**, then **Runtime → Run all** again.


## 1. Environment setup

Clones the repo (public GitHub remote) and installs the pinned package versions from `ml/requirements.txt`, so the notebook environment matches the project's `.venv` exactly.


In [ ]:
import os

REPO_URL = "https://github.com/Terrymanzi/MBERE_ML.git"
REPO_DIR = "/content/MBERE_ML"

if not os.path.isdir(REPO_DIR):
    !git clone --quiet {REPO_URL} {REPO_DIR}
else:
    print("Repo already present — pulling latest changes.")
    !git -C {REPO_DIR} pull --quiet

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())

!pip install -q -r ml/requirements.txt

In [ ]:
import sys

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

import matplotlib
import numpy as np
import pandas as pd
import sklearn

print("python  ", sys.version.split()[0])
print("pandas  ", pd.__version__)
print("numpy   ", np.__version__)
print("sklearn ", sklearn.__version__)

import xgboost
import imblearn
import shap
print("xgboost ", xgboost.__version__)
print("imblearn", imblearn.__version__)
print("shap    ", shap.__version__)

!git -C {REPO_DIR} rev-parse --short HEAD

## 2. Notebook plotting style

A small, fixed color system so every chart in this notebook reads consistently:
- **Categorical** (model identity: baseline / random_forest / xgboost) — fixed hue order, never re-cycled.
- **Ordinal** (severity: Slight → Serious → Fatal) — a single blue ramp, light → dark, since severity is an *ordered* magnitude, not an unordered category.


In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Image, display

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": "#c3c2b7",
    "axes.labelcolor": "#0b0b0b",
    "text.color": "#0b0b0b",
    "xtick.color": "#52514e",
    "ytick.color": "#52514e",
    "axes.grid": True,
    "grid.color": "#e1e0d9",
    "grid.linewidth": 0.8,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

# Fixed categorical order (never re-cycled across charts): baseline, random_forest, xgboost
MODEL_COLORS = {
    "baseline": "#2a78d6",       # blue
    "random_forest": "#1baf7a",  # aqua
    "xgboost": "#eda100",        # yellow
}
MODEL_ORDER = ["baseline", "random_forest", "xgboost"]

# Ordinal severity ramp (light -> dark blue), start no lighter than step 250
SEVERITY_COLORS = ["#86b6ef", "#2a78d6", "#104281"]  # Slight, Serious, Fatal

# Categorical pair for before/after or train/test comparisons
PAIR_COLORS = ["#2a78d6", "#1baf7a"]

def bar_with_labels(ax, x, heights, colors, fmt="{:.3f}"):
    bars = ax.bar(x, heights, color=colors, width=0.6)
    for rect, h in zip(bars, heights):
        ax.annotate(fmt.format(h), (rect.get_x() + rect.get_width() / 2, h),
                    ha="center", va="bottom", fontsize=9, color="#52514e")
    return bars

def fmt4(x):
    return f"{x:.4f}" if x is not None else "n/a"

print("Plotting style ready.")

## 3. Load the dataset configuration

Everything downstream — column names, cleaning rules, engineered features, encoding strategy, split, feature-selection threshold — is driven by `ml/configs/addis.yaml`. Nothing here is hardcoded twice; the notebook reads the same config the pipeline scripts do.


In [ ]:
from ml.utils.config import load_config
from ml.utils.paths import PROJECT_ROOT

config = load_config(PROJECT_ROOT / "ml" / "configs" / "addis.yaml")

print(f"dataset        : {config.name} ({config.kind})")
print(f"target column  : {config.target.column}")
print(f"classes        : {config.target.classes}")
print(f"raw path       : {config.paths.raw}")
print(f"test_size      : {config.split.test_size}  (stratified={config.split.stratify})")
print(f"k_folds        : {config.split.k_folds}")
print(f"feature_select : {config.feature_selection.method} @ threshold={config.feature_selection.threshold}")
print(f"baseline kind  : {config.baseline.kind}")
print(f"n features     : {len(config.features.all)} -> {config.features.all}")

## 4. Raw data exploration

Before any cleaning: shape, dtypes, and the raw (uncleaned) target distribution — including the inconsistent `"Fatal injury"` casing that `clean()` will normalize.


In [ ]:
from ml.preprocessing.load import load_raw

raw_df = load_raw(config)
print("raw shape:", raw_df.shape)
raw_df.head()

In [ ]:
print(raw_df.dtypes)
print()
print("Raw target value counts (before cleaning / label normalization):")
print(raw_df[config.target.column].value_counts(dropna=False))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
counts = raw_df[config.target.column].value_counts(dropna=False)
bar_with_labels(ax, range(len(counts)), counts.values,
                colors=[SEVERITY_COLORS[i % 3] for i in range(len(counts))], fmt="{:.0f}")
ax.set_xticks(range(len(counts)))
ax.set_xticklabels(counts.index, rotation=20, ha="right")
ax.set_ylabel("rows")
ax.set_title("Raw target distribution (pre-cleaning)")
fig.tight_layout()
plt.show()

## 5. Data cleaning

`ml.preprocessing.clean.clean()` is deterministic and row-wise (no statistic is learned from the data), so it is safe to run on the full dataset before splitting:

- drops **leakage columns** (post-accident casualty/outcome fields — `Casualty_severity` is effectively a target proxy)
- derives `hour_of_day` from the raw `Time` column, then drops the raw timestamp
- trims strings and maps missing tokens (`""`, `"na"`, `"nan"`, `"unknown"`) to a constant fill value (`"Unknown"`) — parameter-free, so leakage-safe
- canonicalizes target labels (e.g. `"Fatal injury"` → `"Fatal Injury"`) and drops rows with an invalid/missing target or unparseable time


In [ ]:
from ml.preprocessing.clean import clean

n_before = len(raw_df)
cleaned_df = clean(raw_df, config)
n_after = len(cleaned_df)

print(f"rows: {n_before} -> {n_after}  (dropped {n_before - n_after})")
print(f"columns after cleaning: {cleaned_df.shape[1]}")
cleaned_df.head()

In [ ]:
print("Any missing values left in non-target columns?")
non_target = cleaned_df.drop(columns=[config.target.column])
print(non_target.isna().sum().sum(), "missing cells (expected 0 -- constant-filled)")
print()
print("Cleaned target distribution:")
print(cleaned_df[config.target.column].value_counts())

## 6. Feature engineering

`ml.features.feature_engineering.engineer_features()` derives the proposal's interpretable features from the cleaned columns — still deterministic and row-wise, so still safe before the split:

- 1:1 renames (e.g. `Age_band_of_driver` → `driver_age_band`)
- `time_of_day` bucket derived from `hour_of_day` (Night / Morning / Afternoon / Evening)
- `vehicle_type` grouping from the 17 raw vehicle labels via ordered keyword rules (motorcycle kept first-class, not folded into "Other")


In [ ]:
from ml.features.feature_engineering import engineer_features

engineered_df = engineer_features(cleaned_df, config)
print("engineered shape:", engineered_df.shape)
print("engineered feature columns:", [c for c in engineered_df.columns if c != config.target.column])
engineered_df.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

for ax, col, title in zip(
    axes, ["time_of_day", "vehicle_type"],
    ["Derived: time_of_day", "Derived: vehicle_type (grouped)"],
):
    counts = engineered_df[col].value_counts()
    ax.barh(counts.index[::-1], counts.values[::-1], color="#2a78d6")
    ax.set_title(title)
    ax.set_xlabel("rows")

fig.tight_layout()
plt.show()

## 7. Train/test split (stratified)

`ml.preprocessing.split.make_split()` returns reproducible, **stratified** positional indices — this preserves class proportions (especially the rare `Fatal` class) in both folds. The split happens *before* any fitting (encoder, feature selection), so nothing downstream ever sees test data during fitting.


In [ ]:
from ml.preprocessing.split import make_split

train_idx, test_idx = make_split(engineered_df, config)
train_df = engineered_df.iloc[train_idx].reset_index(drop=True)
test_df = engineered_df.iloc[test_idx].reset_index(drop=True)

print(f"train: {len(train_df)} rows   test: {len(test_df)} rows "
      f"({len(test_df) / len(engineered_df):.1%} test)")

In [ ]:
train_props = train_df[config.target.column].value_counts(normalize=True).reindex(config.target.classes)
test_props = test_df[config.target.column].value_counts(normalize=True).reindex(config.target.classes)

x = np.arange(len(config.target.classes))
width = 0.35
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.bar(x - width / 2, train_props.values, width, label="train", color=PAIR_COLORS[0])
ax.bar(x + width / 2, test_props.values, width, label="test", color=PAIR_COLORS[1])
ax.set_xticks(x)
ax.set_xticklabels(config.target.classes)
ax.set_ylabel("class proportion")
ax.set_title("Stratification check: train vs. test class proportions")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

## 8. Feature selection (mutual information, TRAIN only)

`ml.features.feature_selection.select_features()` scores each engineered feature against the target **using only the training fold**, and keeps features scoring at or above the config threshold (`0.0003`). A guard never drops every feature.


In [ ]:
from ml.features.feature_selection import compute_scores, select_features

scores = compute_scores(train_df[config.features.all], train_df[config.target.column], config)
selected, _ = select_features(train_df[config.features.all], train_df[config.target.column], config)

scores_sorted = dict(sorted(scores.items(), key=lambda kv: kv[1], reverse=True))
colors = ["#1baf7a" if f in selected else "#e34948" for f in scores_sorted]

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(list(scores_sorted.keys())[::-1], list(scores_sorted.values())[::-1], color=colors[::-1])
ax.axvline(config.feature_selection.threshold, color="#898781", linestyle="--", linewidth=1,
           label=f"threshold = {config.feature_selection.threshold}")
ax.set_xlabel("mutual information score")
ax.set_title("Feature selection scores (TRAIN only) -- green = kept, red = dropped")
ax.legend(frameon=False, loc="lower right")
fig.tight_layout()
plt.show()

dropped = [f for f in config.features.all if f not in selected]
print(f"selected {len(selected)}/{len(config.features.all)} features: {selected}")
print(f"dropped: {dropped}")

## 9. Encoding (ordinal + one-hot, fit on TRAIN only)

`ml.preprocessing.encode.build_encoder()` builds an **unfitted** `ColumnTransformer`:
- **ordinal** features use an explicit category order (unseen categories at transform time map to `-1`)
- **one-hot** features use `OneHotEncoder(handle_unknown="ignore")` (unseen categories map to an all-zero block)

It is fit on the training fold only, then applied to test — categories and scaling never leak.


In [ ]:
from ml.preprocessing.encode import build_encoder, encoded_feature_names, fit_encoder, transform

encoder = fit_encoder(build_encoder(config, subset=selected), train_df[selected])
X_train_enc = transform(encoder, train_df[selected])
X_test_enc = transform(encoder, test_df[selected])
feature_names_enc = encoded_feature_names(encoder)

print(f"encoded feature dim: {X_train_enc.shape[1]} (from {len(selected)} input features)")
print(f"X_train_enc shape: {X_train_enc.shape}   X_test_enc shape: {X_test_enc.shape}")
pd.DataFrame(X_train_enc[:5], columns=feature_names_enc)

## 10. Run the authoritative preprocessing pipeline

The steps above (5-9) were run individually to make each stage visible. `ml.preprocessing.preprocess.run()` performs the exact same `clean -> engineer -> split -> select -> encode` sequence as the single source of truth, and **persists** the artifacts the rest of the pipeline (training, evaluation, SHAP) reads from disk:

- `data/processed/{name}_train.csv`, `{name}_test.csv` — engineered, selected features + target
- `data/processed/{name}_train_encoded.npz`, `{name}_test_encoded.npz` — model-ready `X`, `y`
- `ml/artifacts/encoders.joblib` — the fitted `ColumnTransformer`
- `ml/artifacts/split_indices.json` — reproducible train/test indices
- `ml/artifacts/feature_contract.json` — ordered feature names, dtypes, categories (the model input contract)


In [ ]:
from ml.preprocessing.preprocess import run as run_preprocessing

result = run_preprocessing(config)

print(f"train rows : {result.n_train}")
print(f"test rows  : {result.n_test}")
print(f"selected   : {result.selected_features}")
print(f"train csv  : {result.train_path}")
print(f"test csv   : {result.test_path}")
print(f"encoder    : {result.encoder_path}")
print(f"contract   : {result.contract_path}")

## 11. Model architecture

Three classifiers are trained and compared through an **identical harness** (`ml.training.common`), so results are directly comparable:

| Model | Architecture | Notes |
|---|---|---|
| `baseline` | `RuleBasedRiskClassifier` | Transparent additive risk-scoring rules (young/inexperienced drivers, night + poor lighting, adverse weather, vulnerable vehicles, poor road surface) mapped to an ordinal cumulative-probability formulation over the 3 severity classes. **Not** a dummy classifier — this is the bar the ML models must clear. |
| `random_forest` | `encoder -> SMOTE -> RandomForestClassifier` (imblearn `Pipeline`) | 300 trees, `min_samples_leaf=2`, single-threaded (`n_jobs=1`) for bit-exact reproducibility of `predict_proba`. |
| `xgboost` | `encoder -> SMOTE -> XGBClassifier` | 400 trees, `max_depth=6`, `learning_rate=0.1`, `multi:softprob` objective (kind-aware: switches to `binary:logistic` for binary configs). |

**Leak-free guarantees the harness enforces:**
- All three models are evaluated with the **same** `StratifiedKFold` splits (shared `random_state`) via out-of-fold (OOF) predictions.
- **SMOTE lives inside the pipeline**, so it resamples *only* the training fold on each `fit()` call and is a no-op at `predict()` time — the validation/test fold is never oversampled.


### 11.1 SMOTE demonstration (train-fold-only resampling)

To make the leak-free guarantee concrete: take one CV fold, encode it, and compare class counts before vs. after SMOTE. This uses the *same* `build_encoder` the training pipeline uses — nothing reimplemented.


In [ ]:
import json

from imblearn.over_sampling import SMOTE

from ml.training.common import get_cv, load_processed

X_train_full, y_train_full, classes = load_processed(config, "train")
cv = get_cv(config)
fold_train_idx, fold_val_idx = next(cv.split(X_train_full, y_train_full))

fold_encoder = build_encoder(config, subset=selected)
X_fold_enc = fold_encoder.fit_transform(X_train_full.iloc[fold_train_idx][selected])
y_fold = y_train_full[fold_train_idx]

smote = SMOTE(random_state=config.random_state)
X_res, y_res = smote.fit_resample(X_fold_enc, y_fold)

before = pd.Series(y_fold).value_counts().reindex(range(len(classes)), fill_value=0)
after = pd.Series(y_res).value_counts().reindex(range(len(classes)), fill_value=0)

x = np.arange(len(classes))
width = 0.35
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.bar(x - width / 2, before.values, width, label="before SMOTE", color=PAIR_COLORS[0])
ax.bar(x + width / 2, after.values, width, label="after SMOTE", color=PAIR_COLORS[1])
ax.set_xticks(x)
ax.set_xticklabels(classes)
ax.set_ylabel("training-fold rows")
ax.set_title("SMOTE rebalances the TRAINING fold only (this fold, this notebook)")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

print("Validation fold is untouched by SMOTE -- val class counts:")
print(pd.Series(y_train_full[fold_val_idx]).value_counts().reindex(range(len(classes)), fill_value=0))

## 12. Model training

Each model is trained with `ml.training.common.train_and_save()`:
1. out-of-fold (OOF) predictions across the shared `StratifiedKFold`, scored with `compute_metrics` (headline: macro F1, macro recall, macro ROC-AUC one-vs-rest; accuracy is reported but **not** headline given class imbalance)
2. a final model refit on **all** training data
3. the fitted model + a versioned metadata sidecar (git commit, random_state, CV scheme, params, CV metrics) saved to `ml/artifacts/`


### 12.1 Baseline — rule-based risk classifier

In [ ]:
from ml.models import make_baseline
from ml.training.common import estimator_params, train_and_save

baseline_estimator = make_baseline(config)
baseline_path, baseline_meta_path, baseline_cv_metrics = train_and_save(
    config, "baseline", "0.1.0", baseline_estimator, estimator_params(baseline_estimator)
)
print(f"f1_macro={baseline_cv_metrics['f1_macro']:.4f}  "
      f"recall_macro={baseline_cv_metrics['recall_macro']:.4f}  "
      f"roc_auc_ovr_macro={fmt4(baseline_cv_metrics['roc_auc_ovr_macro'])}")

### 12.2 Random Forest (SMOTE + RF)

In [ ]:
from ml.training.train_rf import build_estimator as build_rf_estimator

rf_estimator = build_rf_estimator(config)
rf_params = rf_estimator.named_steps["classifier"].get_params()
rf_path, rf_meta_path, rf_cv_metrics = train_and_save(
    config, "random_forest", "0.1.0", rf_estimator, rf_params
)
print(f"f1_macro={rf_cv_metrics['f1_macro']:.4f}  "
      f"recall_macro={rf_cv_metrics['recall_macro']:.4f}  "
      f"roc_auc_ovr_macro={fmt4(rf_cv_metrics['roc_auc_ovr_macro'])}")

### 12.3 XGBoost (SMOTE + XGBoost)

In [ ]:
from ml.training.train_xgboost import build_estimator as build_xgb_estimator

xgb_estimator = build_xgb_estimator(config)
xgb_params = xgb_estimator.named_steps["classifier"].get_params()
xgb_path, xgb_meta_path, xgb_cv_metrics = train_and_save(
    config, "xgboost", "0.1.0", xgb_estimator, xgb_params
)
print(f"f1_macro={xgb_cv_metrics['f1_macro']:.4f}  "
      f"recall_macro={xgb_cv_metrics['recall_macro']:.4f}  "
      f"roc_auc_ovr_macro={fmt4(xgb_cv_metrics['roc_auc_ovr_macro'])}")

## 13. Cross-validation metrics comparison

Out-of-fold metrics for all three models, side by side (fixed model colors: baseline = blue, random_forest = aqua, xgboost = yellow).


In [ ]:
cv_metrics_by_model = {
    "baseline": baseline_cv_metrics,
    "random_forest": rf_cv_metrics,
    "xgboost": xgb_cv_metrics,
}

cv_summary = pd.DataFrame({
    model: {
        "f1_macro": m["f1_macro"],
        "recall_macro": m["recall_macro"],
        "precision_macro": m["precision_macro"],
        "roc_auc_ovr_macro": m["roc_auc_ovr_macro"],
        "accuracy": m["accuracy"],
    }
    for model, m in cv_metrics_by_model.items()
}).T.loc[MODEL_ORDER]
display(cv_summary.round(4))

metrics_to_plot = ["f1_macro", "recall_macro", "roc_auc_ovr_macro"]
x = np.arange(len(metrics_to_plot))
width = 0.25
fig, ax = plt.subplots(figsize=(8, 4.5))
for i, model in enumerate(MODEL_ORDER):
    values = [cv_summary.loc[model, m] for m in metrics_to_plot]
    ax.bar(x + (i - 1) * width, values, width, label=model, color=MODEL_COLORS[model])
ax.set_xticks(x)
ax.set_xticklabels(metrics_to_plot)
ax.set_ylim(0, 1)
ax.set_title("Cross-validation (out-of-fold) metrics by model")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

## 14. Held-out test set evaluation

`ml.evaluation.evaluate.evaluate_model()` loads each saved model, predicts on the untouched test set, and writes `confusion_matrix.png`, `roc_curves.png`, and `metrics.json` under `ml/artifacts/reports/<model>/`.


In [ ]:
from ml.evaluation.evaluate import evaluate_model

test_metrics_by_model = {}
for model_name in MODEL_ORDER:
    print(f"\n=== {model_name} ===")
    metrics = evaluate_model(config, model_name)
    test_metrics_by_model[model_name] = metrics
    print(f"f1_macro={metrics['f1_macro']:.4f}  recall_macro={metrics['recall_macro']:.4f}  "
          f"roc_auc_ovr_macro={fmt4(metrics['roc_auc_ovr_macro'])}  accuracy={metrics['accuracy']:.4f}")

    report_dir = config.paths.artifacts_dir / "reports" / model_name
    display(Image(filename=str(report_dir / "confusion_matrix.png")))
    display(Image(filename=str(report_dir / "roc_curves.png")))

## 15. Test metrics comparison & model selection

Same comparison as the CV chart above, now on the held-out test set — the number that matters for model selection.


In [ ]:
test_summary = pd.DataFrame({
    model: {
        "f1_macro": m["f1_macro"],
        "recall_macro": m["recall_macro"],
        "precision_macro": m["precision_macro"],
        "roc_auc_ovr_macro": m["roc_auc_ovr_macro"],
        "accuracy": m["accuracy"],
    }
    for model, m in test_metrics_by_model.items()
}).T.loc[MODEL_ORDER]
display(test_summary.round(4))

x = np.arange(len(metrics_to_plot))
width = 0.25
fig, ax = plt.subplots(figsize=(8, 4.5))
for i, model in enumerate(MODEL_ORDER):
    values = [test_summary.loc[model, m] for m in metrics_to_plot]
    ax.bar(x + (i - 1) * width, values, width, label=model, color=MODEL_COLORS[model])
ax.set_xticks(x)
ax.set_xticklabels(metrics_to_plot)
ax.set_ylim(0, 1)
ax.set_title("Held-out test-set metrics by model")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

best_model = test_summary["f1_macro"].idxmax()
print(f"Best model by test f1_macro: '{best_model}' "
      f"(f1_macro={test_summary.loc[best_model, 'f1_macro']:.4f})")

## 16. Explainability -- SHAP

`ml.explainability.shap_analysis.analyze()` runs `shap.TreeExplainer` on the tree models (Random Forest, XGBoost) using the encoded test features, and writes:
- `summary_plot.png` — global feature impact, one-vs-rest per class
- `waterfall_plot.png` — local explanation for the single highest-predicted-risk (most-severe-class) case in the sample
- `importance.csv` — mean `|SHAP|` per encoded feature, descending

The rule-based baseline is skipped here since its "explanation" is simply its fixed, human-readable rule set (Section 11) — SHAP applies to the two opaque tree models.


In [ ]:
from ml.explainability.shap_analysis import analyze as shap_analyze

shap_summaries = {}
for model_name in ["random_forest", "xgboost"]:
    print(f"\n=== SHAP: {model_name} ===")
    summary = shap_analyze(config, model_name, sample_size=500)
    shap_summaries[model_name] = summary

    report_dir = config.paths.artifacts_dir / "reports" / model_name
    display(Image(filename=str(report_dir / "summary_plot.png")))
    display(Image(filename=str(report_dir / "waterfall_plot.png")))

    importance = pd.read_csv(report_dir / "importance.csv")
    print(f"Top 10 features by mean |SHAP| ({model_name}):")
    display(importance.head(10))

## 17. Saved artifacts & reproducibility

Every artifact below is versioned with its provenance (git commit, random_state, CV scheme) in its `.meta.json` sidecar, so any result in this notebook can be traced back to the exact code + data that produced it.


In [ ]:
artifacts_dir = config.paths.artifacts_dir
for path in sorted(artifacts_dir.rglob("*")):
    if path.is_file():
        print(path.relative_to(artifacts_dir))

In [ ]:
with open(config.paths.artifacts_dir / "random_forest.meta.json") as fh:
    rf_meta = json.load(fh)

print("random_forest.meta.json (provenance):")
print(f"  git_commit  : {rf_meta['git_commit']}")
print(f"  random_state: {rf_meta['random_state']}")
print(f"  cv          : {rf_meta['cv']}")
print(f"  dataset     : {rf_meta['dataset']}")

In [ ]:
# Reload the best model from disk and predict on a single held-out test row, end to end.
from ml.utils.artifacts import load_model

reloaded = load_model(config.paths.artifacts_dir / f"{best_model}.pkl")
sample_row = test_df[selected].iloc[[0]]
pred_idx = int(reloaded.predict(sample_row)[0])
pred_proba = reloaded.predict_proba(sample_row)[0]

print("Sample input features:")
display(sample_row)
print(f"Predicted severity: {config.target.classes[pred_idx]}")
print(f"Class probabilities: {dict(zip(config.target.classes, np.round(pred_proba, 4)))}")
print(f"Actual severity: {test_df[config.target.column].iloc[0]}")

## 18. Optional: synthetic (Rwandan) data sanity check

`ml.synthetic.generate` validates a **Fabricate-generated** synthetic Rwandan driver-risk CSV against the real feature vocabulary (`feature_contract.json`) and runs a contextual monotonicity check (mean P(Fatal) / P(Serious) should increase with the synthetic dataset's own severity label).

> **This is context/robustness validation only** — it is never merged into `metrics.json` and never used as performance ground truth; the Addis held-out test set (Sections 14-15) remains the sole source of performance metrics.

This section needs an external CSV that is **not** part of the repository (it's generator-tool output, not tracked in git). Upload one to `data/external/<name>.csv` in the Colab file browser to run it; otherwise this cell skips gracefully.


In [ ]:
from pathlib import Path

external_dir = Path("data/external")
synthetic_csvs = sorted(external_dir.glob("*.csv")) if external_dir.is_dir() else []

if not synthetic_csvs:
    print("No synthetic CSV found under data/external/ -- skipping.")
    print("To run this section: upload a Fabricate-generated CSV "
          "(must include a 'synthetic_severity_label' column) to data/external/, then re-run this cell.")
else:
    from ml.synthetic.generate import load_and_validate, sanity_check

    synthetic_path = synthetic_csvs[0]
    print(f"Found {synthetic_path} -- validating against the feature contract...")

    synthetic_df = load_and_validate(synthetic_path, config.paths.artifacts_dir / "feature_contract.json")
    print(f"Validated {len(synthetic_df)} synthetic rows.")

    result = sanity_check(
        synthetic_df,
        model_path=config.paths.artifacts_dir / f"{best_model}.pkl",
        encoder_path=config.paths.artifacts_dir / "encoders.joblib",
        contract_path=config.paths.artifacts_dir / "feature_contract.json",
    )
    print(json.dumps(result, indent=2))

## 19. Download artifacts from Colab

Zips `ml/artifacts/` (models, metadata, reports, SHAP plots) and `data/processed/` (encoded train/test sets) for download. This only works when running on Colab.


In [ ]:
import shutil

shutil.make_archive("/content/ml_artifacts", "zip", "ml/artifacts")
shutil.make_archive("/content/processed_data", "zip", "data/processed")

try:
    from google.colab import files
    files.download("/content/ml_artifacts.zip")
    files.download("/content/processed_data.zip")
except ImportError:
    print("Not running on Colab -- archives written to /content/*.zip")

## Conclusion

This notebook reproduced the full MBERE ML pipeline end to end on a fresh Colab runtime:

- **Data**: Addis Ababa RTA data, cleaned leakage-safe and deterministically, with 11-13 interpretable engineered features selected by train-only mutual information.
- **Models**: a transparent rule-based baseline, a Random Forest, and an XGBoost classifier, all trained through the same leak-free `StratifiedKFold` + SMOTE-in-pipeline harness.
- **Evaluation**: out-of-fold CV metrics and held-out test metrics agree closely (see Sections 13 & 15), with macro F1 / recall / ROC-AUC as headline metrics given class imbalance.
- **Explainability**: SHAP summary and waterfall plots show which engineered features (e.g. driver age band, experience, night + poor lighting, vehicle type) drive severity predictions for both tree models.
- **Reproducibility**: every saved model carries a metadata sidecar with its git commit, random seed, and CV scheme, so any number in this notebook can be traced back to exact code + data.

**Next steps:** wire the winning model (Section 15) into the backend inference service via `feature_contract.json`; extend `ml/configs/` with additional dataset configs as more labeled data becomes available; consider adding a hyperparameter-search step (not currently part of the pipeline) if CV/test metrics plateau.
